# 00 — Load Bronze Data

**J&J HRD 2030 Predictive Hiring Demo**

This notebook bootstraps the Unity Catalog infrastructure and ingests raw source data into Bronze Delta tables.

### What this notebook does
1. Creates the UC Catalog, Schema, and Volume if they don't already exist
2. Uploads structured CSV files (candidates, job requirements, training data) to the Volume
3. Uploads 10 PDF resumes to the Volume under a `resumes/` subfolder
4. Creates Bronze Delta tables from the CSVs using `read_files()`
5. Validates row counts for each table

### Prerequisites
- Unity Catalog enabled workspace
- Cluster with appropriate UC permissions (`CREATE CATALOG`, `CREATE SCHEMA`, etc.)
- Source files present under `data/` relative to this notebook in the workspace repo

**Next notebook:** `01_load_silver.ipynb`

In [ ]:
%pip install pypdf -q

## Setup — Configuration Parameters

Define the Unity Catalog hierarchy (catalog, schema, volume) via widgets so this notebook can be parameterized at runtime or in a job.

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions — edit defaults here or override at runtime via the UI
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",     "bx4",          "UC Catalog")
dbutils.widgets.text("schema",      "hrd_2030",     "UC Schema")
dbutils.widgets.text("volume_name", "hrd_raw_data", "UC Volume Name")

In [ ]:
catalog     = dbutils.widgets.get("catalog")
schema      = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

print(f"Catalog     : {catalog}")
print(f"Schema      : {schema}")
print(f"Volume      : {volume_name}")
print(f"Volume path : {volume_path}")

## Verify Volume Contents

Confirm the Unity Catalog Volume is accessible and that the expected CSV and PDF files have been pre-uploaded via the Databricks CLI.

In [ ]:
# Schema and Volume are pre-created and files pre-uploaded via CLI.
# Just verify the volume is accessible.
files = [f.name for f in dbutils.fs.ls(volume_path)]
print(f"✓ Volume accessible. Contents: {files}")

In [ ]:
# Files were pre-uploaded to the volume via the Databricks CLI.
# Verify CSVs are present. training_data.csv has been retired —
# feature scores and hired label are now embedded in candidates.csv
csv_files = ["candidates.csv", "job_requirements.csv"]
for fname in csv_files:
    path = f"{volume_path}/{fname}"
    try:
        dbutils.fs.ls(path)
        print(f"  ✓ {fname}")
    except Exception:
        print(f"  ✗ MISSING: {fname}")

In [ ]:
resumes_path = f"{volume_path}/resumes"
pdfs = [f.name for f in dbutils.fs.ls(resumes_path)]
print(f"✓ {len(pdfs)} PDFs in volume: {pdfs}")

## Create Bronze Delta Tables

Ingest raw CSV files from the Volume into Bronze Delta tables using `read_files()`. All columns are stored as STRING — type casting happens in the Silver notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Bronze table: bronze_candidates
# All columns ingested as STRING — type-casting happens in the Silver notebook
# ---------------------------------------------------------------------------
candidates_csv = f"{volume_path}/candidates.csv"

spark.sql(f"""
  CREATE OR REPLACE TABLE `{catalog}`.`{schema}`.bronze_candidates
  COMMENT 'Raw candidate data ingested from CSV via Volume. All columns stored as STRING.'
  AS
  SELECT *
  FROM read_files(
      '{candidates_csv}',
      format  => 'csv',
      header  => 'true',
      inferSchema => 'false'
  )
""")

print("✓ bronze_candidates created.")

In [ ]:
# ---------------------------------------------------------------------------
# Bronze table: bronze_job_requirements
# ---------------------------------------------------------------------------
job_req_csv = f"{volume_path}/job_requirements.csv"

spark.sql(f"""
  CREATE OR REPLACE TABLE `{catalog}`.`{schema}`.bronze_job_requirements
  COMMENT 'Raw job requirements ingested from CSV via Volume. All columns stored as STRING.'
  AS
  SELECT *
  FROM read_files(
      '{job_req_csv}',
      format  => 'csv',
      header  => 'true',
      inferSchema => 'false'
  )
""")

print("✓ bronze_job_requirements created.")

## Validate Row Counts

Confirm each Bronze table was created successfully by printing row counts.

In [ ]:
# ---------------------------------------------------------------------------
# Row-count verification for all Bronze tables
# ---------------------------------------------------------------------------
bronze_tables = [
    "bronze_candidates",
    "bronze_job_requirements",
]

print(f"{'Table':<35} {'Row Count':>10}")
print("-" * 47)
for tbl in bronze_tables:
    full_name = f"`{catalog}`.`{schema}`.{tbl}"
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {full_name}").collect()[0]["n"]
    print(f"{tbl:<35} {count:>10,}")

print("\n✓ Bronze validation complete.")

## Next Steps

Bronze tables are now available in `{catalog}.{schema}`:

| Table | Contents |
|---|---|
| `bronze_candidates` | Candidate profiles (all STRING columns) |
| `bronze_job_requirements` | Open HR Director requisitions (all STRING columns) |
| `bronze_training_data` | Historical hiring outcomes for model training (all STRING columns) |

PDF resumes have been staged in the Volume under `/resumes/` for downstream text extraction.

**Proceed to** `01_load_silver.ipynb` to cast types, clean data, and produce Silver tables.